# Foundry IQ knowledge base with Fabric IQ integration

This notebook creates a Foundry IQ knowledge base with a **Fabric Ontology Knowledge Source** that connects to structured operational data in Microsoft Fabric.

- **Fabric Ontology Knowledge Source**: connects to a Fabric workspace ontology. The KB issues structured queries against entity types like `Product` at runtime.
- **Delegated token**: This notebook uses `InteractiveBrowserCredential` to sign in the current user and obtain a delegated token for Azure AI Search. Search uses that delegated user identity when querying protected sources such as Fabric Ontology.

## Step 1: Set up environment variables

Load the configuration for your Azure AI Search, Azure OpenAI, and Fabric resources. The `.env` file in the project folder has the values needed for this part.

In [1]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
FABRIC_TENANT_ID = os.environ["FABRIC_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


Environment variables loaded


## Step 2: Login and acquire delegated token

In order for the knowledge base retrieval call to use your identity for query source access, it needs an OAuth token for a signed-in user who has access to the Fabric workspace.

Run the cell below to sign in to Azure using the credentials in the instructions sidebar. If successful, it acquires a delegated token for the Azure AI Search scope and saves it for use as `query_source_authorization` during retrieval.

In [2]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(tenant_id=FABRIC_TENANT_ID)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

/Users/pamelafox/iqdeepdive-foundryiq-1/.venv/lib/python3.14/site-packages/msal/oauth2cli/oauth2.py:453: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(


Acquired token for logged-in user


## Step 3: Create three knowledge sources

For this knowledge base, you'll create three knowledge sources: the same two search-index sources used earlier, plus a Fabric Ontology knowledge source.

* `hrdocs-knowledge-source`: Points to the `hrdocs` search index
* `healthdocs-knowledge-source`: Points to the `healthdocs` search index
* `fabric-ontology-knowledge-source`: Points to the Fabric workspace and ontology used for structured operational data

In [3]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    FabricOntologyKnowledgeSource,
    FabricOntologyKnowledgeSourceParameters,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
FABRIC_KNOWLEDGE_SOURCE_NAME = "fabric-ontology-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    FABRIC_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Contoso HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Contoso health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

fabric_knowledge_source = FabricOntologyKnowledgeSource(
    name=FABRIC_KNOWLEDGE_SOURCE_NAME,
    description="Contoso Fabric Ontology knowledge source with product data",
    fabric_ontology_parameters=FabricOntologyKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        ontology_id=FABRIC_ONTOLOGY_ID,
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=fabric_knowledge_source)
print("Knowledge sources created")


Knowledge sources created


## Step 4: Create the multi-source + Fabric knowledge base

A knowledge base combines the knowledge sources with an LLM and instructions for how retrieval should behave.

For this notebook, the knowledge base uses an `outputMode` of `answerSynthesis` so the attached model can answer the question directly. It also uses a `retrievalReasoningEffort` of `low` so the model can help with query planning and source selection.

In [4]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-fabric-ontology-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base combining indexed company documents and Contoso structured product data",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Fabric Ontology for structured operational data. Use the search indexes for HR and health policy documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


Knowledge base created


## Step 5: Query the knowledge base

Ask a question that combines structured Fabric Ontology data with reasoning from the attached chat model.

The knowledge base uses agentic retrieval to decide when to query the search indexes and when to query the Fabric Ontology source.

⏱️ The retrieval takes ~40 seconds to complete, due to the increased latency from the Fabric IQ knowledge source.

In [5]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricOntologyKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

question = ("I'm a Contoso buyer prepping for the summer sale. "
            "Which product categories have the lowest stock levels right now? "
            "Which Contoso job roles are responsible for inventory strategy and budget oversight?")

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        FabricOntologyKnowledgeSourceParams(
            knowledge_source_name=FABRIC_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
            reranker_threshold=2.0
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=180,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)
display(Markdown(result.response[0].content[0].text))


Right now, the product categories with the lowest stock levels are: Hand Tools[ref_id:2], Lumber & Building Materials[ref_id:2], Electrical[ref_id:2], Power Tools[ref_id:2], Paint & Finishes[ref_id:2], Hardware[ref_id:2], Garden & Outdoor[ref_id:2], Storage & Organization[ref_id:2], and Plumbing[ref_id:2].

The inventory data specifically shows multiple zero-unit items in each of those categories, which is why they are identified as the lowest-stock categories at the moment[ref_id:2].

For Contoso job roles tied to inventory strategy and budget oversight, the clearest matches I found are:

- Director of Operations, which includes developing and implementing strategies to manage inventory and stock levels[ref_id:0] and managing the budget and financial operations of the company[ref_id:0].
- Vice President of Operations, which includes developing and executing strategic operational plans[ref_id:10] and overseeing operational plans and budgets[ref_id:10].
- Senior Manager of Operations, which oversees supply chain, logistics, and finance[ref_id:5] and develops, implements, and monitors operational plans, budgets, and performance goals[ref_id:5].
- Manager of Operations, which develops operational strategies[ref_id:16] and develops and manages operational budgets while monitoring financial performance[ref_id:16].
- Chief Operating Officer, which develops and monitors organizational strategies, policies, and procedures[ref_id:7] and manages the budget to keep the organization operating within its means[ref_id:7].

If you want the single best role match for both responsibilities together, Director of Operations is the strongest fit because it explicitly covers both inventory/stock-level strategy[ref_id:0] and budget management[ref_id:0].

### Review the activity log

For this activity log, you will see a "fabricOntology" step for the call made to Fabric IQ, along with "searchIndex" steps for each query sent to a search index.

In [6]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "inputTokens": 1491,
    "outputTokens": 111,
    "modelName": "gpt-5.4",
    "elapsedMs": 1951
  },
  {
    "type": "searchIndex",
    "id": 1,
    "knowledgeSourceName": "hrdocs-knowledge-source",
    "queryTime": "2026-07-24T00:21:44.0157881Z",
    "count": 36,
    "elapsedMs": 0,
    "searchIndexArguments": {
      "search": "inventory strategy budget oversight job roles",
      "filter": null,
      "semanticConfigurationName": "semantic-configuration",
      "sourceDataFields": [
        {
          "name": "snippet"
        },
        {
          "name": "uid"
        },
        {
          "name": "snippet_parent_id"
        },
        {
          "name": "blob_path"
        }
      ],
      "searchFields": [
        {
          "name": "snippet"
        }
      ]
    }
  },
  {
    "type": "searchIndex",
    "id": 3,
    "knowledgeSourceName": "hrdocs-knowledge-source",
    "queryTime": "2026-07-24T00:21:44.2392766Z",
  

### Review the references

For the Fabric IQ knowledge source, the references have `type: "fabricOntology"`. The `sourceData` contains `fabricAnswer` with a synthesized answer and `fabricRawData` with structured data results.

In [7]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

[
  {
    "type": "searchIndex",
    "id": "0",
    "activitySource": 4,
    "sourceData": {
      "uid": "810256edf976_aHR0cHM6Ly9tYWdvdHRlaWFkbHNnZW4yLmJsb2IuY29yZS53aW5kb3dzLm5ldC9sYWJkb2NzL3JvbGVfbGlicmFyeS5wZGY1_pages_14",
      "snippet_parent_id": "aHR0cHM6Ly9tYWdvdHRlaWFkbHNnZW4yLmJsb2IuY29yZS53aW5kb3dzLm5ldC9sYWJkb2NzL3JvbGVfbGlicmFyeS5wZGY1",
      "blob_path": "/hrdocs/role_library.pdf",
      "snippet": "written and verbal communication skills \n\n\u2022 Ability to manage and motivate a team \n\n\u2022 Analytical and problem-solving skills \n\n\u2022 Experience in budget management and financial analysis \n\n\u2022 Knowledge of SEO and web analytics tools \n\n\u2022 Ability to work independently and collaboratively in a fast-paced environment \n\nDirector of Operations \n\nJob Title: Director of Operations, Contoso \n\nJob Summary: \n\nThe Director of Operations is responsible for overseeing the overall operations of Contoso \n\n. This includes determining and implementing 